In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/akarapusreenija/superstore-sales-dataset/SuperStoreOrders.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df["sales"].head(10)

In [ ]:
df["sales"].sample(10, random_state=42)

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns

In [ ]:
df["order_date"].head(20)

In [ ]:
df[df["order_date"].str.contains("-")]["order_date"].head(20)

In [ ]:
# Dates with '/'
df["order_date"].str.contains("/").sum()

In [ ]:
# Dates with '-'
df["order_date"].str.contains("-").sum()

In [ ]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    dayfirst=True
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format="mixed",
    dayfirst=True
)

In [ ]:
df[["order_date", "ship_date"]].dtypes

In [ ]:
df["sales"] = df["sales"].str.replace(",", "")

In [ ]:
df["sales"].sample(10, random_state=42)

In [ ]:
df["sales"] = pd.to_numeric(df["sales"])

In [ ]:
df["sales"].dtype

In [ ]:
df.dtypes

In [ ]:
customers_df = df[
    [
        "customer_name",
        "segment",
        "state",
        "country",
        "market",
        "region",
    ]
].drop_duplicates()

In [ ]:
customers_df.head()

In [ ]:
customers_df.shape

In [ ]:
customers_df = customers_df.reset_index(drop=True)

In [ ]:
customers_df.head()

In [ ]:
customers_df["customer_id"] = [
    f"C{i:05d}" for i in range(1, len(customers_df) + 1)
]

In [ ]:
customers_df.head()

In [ ]:
products_df = df[
    [
        "product_id",
        "category",
        "sub_category",
        "product_name",
    ]
].drop_duplicates()

In [ ]:
products_df = products_df.reset_index(drop=True)

In [ ]:
products_df.head()
products_df.shape

In [ ]:
orders_df = df.merge(
    customers_df[["customer_id", "customer_name"]],
    on="customer_name",
    how="left"
)

In [ ]:
orders_df.head()

In [ ]:
customers_df["customer_name"].duplicated().sum()

In [ ]:
customers_df[customers_df["customer_name"].duplicated(keep=False)].sort_values("customer_name").head(20)

In [ ]:
orders_df = orders_df.drop(
    columns=[
        "product_name",
        "category",
        "sub_category"
    ]
)

In [ ]:
orders_df.head()

In [ ]:
orders_df.columns

In [ ]:
products_df.to_csv("products.csv", index=False)

In [ ]:
orders_df.to_csv("orders.csv", index=False)

In [ ]:
df.shape

In [ ]:
orders_df.shape

In [ ]:
orders_df = df.copy()

In [ ]:
orders_df = orders_df.drop(
    columns=[
        "product_name",
        "category",
        "sub_category"
    ]
)

In [ ]:
orders_df.shape

In [ ]:
products_df.to_csv("products.csv", index=False)
orders_df.to_csv("orders.csv", index=False)

In [ ]:
import os

os.listdir()

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect("retail_sales.db")

In [ ]:
type(conn)

In [ ]:
products_df.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

In [ ]:
pd.read_sql(
    "SELECT * FROM products LIMIT 5;",
    conn
)

In [ ]:
orders_df.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

In [ ]:
pd.read_sql(
    "SELECT * FROM orders LIMIT 5;",
    conn
)

In [ ]:
pd.read_sql(
    """
    SELECT COUNT(*) AS total_products
    FROM products;
    """,
    conn
)

In [ ]:
pd.read_sql(
    """
    SELECT COUNT(*) AS total_orders
    FROM orders;
    """,
    conn
)

In [ ]:
pd.read_sql("""
SELECT SUM(sales) AS total_sales
FROM orders;
""", conn)

In [ ]:
pd.read_sql("""
SELECT AVG(profit) AS average_profit
FROM orders;
""", conn)

In [ ]:
pd.read_sql("""
SELECT MAX(sales) AS highest_sale
FROM orders;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    region,
    SUM(sales) AS total_sales
FROM orders
GROUP BY region
ORDER BY total_sales DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    market,
    COUNT(*) AS total_orders
FROM orders
GROUP BY market
ORDER BY total_orders DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    p.product_name,
    SUM(o.sales) AS total_sales
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_sales DESC
LIMIT 10;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    category,
    SUM(sales) AS total_sales
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY category
ORDER BY total_sales DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    p.category,
    ROUND(SUM(o.profit), 2) AS total_profit
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_profit DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    market,
    ROUND(AVG(discount), 2) AS avg_discount
FROM orders
GROUP BY market
ORDER BY avg_discount DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    p.product_name,
    ROUND(SUM(o.profit), 2) AS total_profit
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_profit DESC
LIMIT 5;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    p.category,
    SUM(o.sales) AS total_sales,
    ROUND(SUM(o.profit), 2) AS total_profit
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    region,
    COUNT(*) AS total_orders
FROM orders
GROUP BY region
HAVING COUNT(*) > 5000
ORDER BY total_orders DESC;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    COUNT(DISTINCT country) AS total_countries
FROM orders;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    order_id,
    profit,
    CASE
        WHEN profit > 0 THEN 'Profitable'
        ELSE 'Loss'
    END AS order_status
FROM orders
LIMIT 10;
""", conn)

In [ ]:
pd.read_sql("""
SELECT
    CASE
        WHEN profit > 0 THEN 'Profitable'
        ELSE 'Loss'
    END AS order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status;
""", conn)